# Screeplot of $L_2$ Loss for AI-READI Dataset

This notebook produces a screeplot of optimal $L_2$ loss values as a function of the number of thresholds $K = 1, 2, \ldots, 12$ on the AI-READI cohort.

**Workflow:**
1. Run `sbatch slurm/screeplot_aireadi.sh` on the cluster to compute `run_de` for each $K$.
2. Results are saved as pickle files in `results/screeplot/`.
3. This notebook loads the results and generates the screeplot.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

np.set_printoptions(precision=4, suppress=True)

## Load results from cluster runs

In [ ]:
cwd = os.getcwd()
results_dir = os.path.join(cwd, 'results', 'screeplot')

K_values = list(range(1, 13))
loss_values = []
cutoff_dict = {}

for K in K_values:
    fpath = os.path.join(results_dir, f'aireadi_Loss2_K{K}.pkl')
    with open(fpath, 'rb') as f:
        res = pickle.load(f)
    loss_values.append(res['min_loss'])
    cutoff_dict[K] = res['best_cutoffs']
    print(f"K={K:2d}  |  Loss2 = {res['min_loss']:.6f}  |  cutoffs = {np.round(res['best_cutoffs'][1:-1], 2)}")

loss_values = np.array(loss_values)

## Screeplot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(K_values, loss_values, 'o-', color='#2c7bb6', markersize=7, linewidth=2, zorder=3)

ax.set_xlabel('Number of thresholds ($K$)', fontsize=13)
ax.set_ylabel('Optimal $L_2$ loss', fontsize=13)
ax.set_title('Screeplot: AI-READI dataset', fontsize=14)

ax.set_xticks(K_values)
ax.xaxis.set_major_locator(ticker.FixedLocator(K_values))
ax.tick_params(labelsize=11)

ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 12.5)

fig.tight_layout()
plt.savefig(os.path.join(cwd, 'results', 'screeplot', 'screeplot_aireadi_Loss2.pdf'),
            bbox_inches='tight', dpi=300)
plt.show()

## Summary table of results

In [ ]:
summary_rows = []
for K in K_values:
    cutoffs_inner = cutoff_dict[K][1:-1]  # exclude endpoints
    summary_rows.append({
        'K': K,
        'Loss2': f"{loss_values[K-1]:.6f}",
        'Cutoffs': ', '.join(f'{c:.2f}' for c in cutoffs_inner),
    })

summary_df = pd.DataFrame(summary_rows).set_index('K')
summary_df